# 🧪 Fine-Tuning XGBoost for Soil pH Prediction with Weights & Biases (W&B) Sweeps
### Bayesian & Grid Hyperparameter Optimization with Spatial Block Cross-Validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Weights & Biases](https://img.shields.io/badge/Weights_&_Biases-FFBE00?logo=weightsandbiases&logoColor=black)](https://wandb.ai/)
[![XGBoost](https://img.shields.io/badge/Model-XGBoost%20Regressor-orange.svg)](https://xgboost.readthedocs.io/)

---

### 📌 Objective
This notebook uses **Weights & Biases (W&B) Sweeps** to systematically search for the optimal set of hyperparameters for predicting **Soil pH** from **Sentinel-2 L2A multispectral reflectance** and derived physical spectral indices.

### 🛡️ Why Spatial Cross-Validation during Fine-Tuning?
1. **Zero Spatial Data Leakage:** Each trial is evaluated by holding out entire geographic regional blocks (`NUTS_2`). This prevents the model from getting falsely inflated scores by memorizing adjacent pixels.
2. **Rigorous Accuracy Evaluation:** After tuning, the notebook computes **R² (Accuracy / Explained Variance), RMSE, MAE (Mean ± Std)**, plots **Actual vs. Predicted scatter diagrams**, and ranks **feature importances**.

## 📦 Step 1: Install Dependencies
Install `wandb`, `xgboost`, `scikit-learn`, `matplotlib`, and data libraries.

In [ ]:
!pip install -q wandb xgboost scikit-learn pandas numpy matplotlib seaborn scipy

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
import wandb

print(f"✅ XGBoost version: {xgb.__version__}")
print(f"✅ W&B version: {wandb.__version__}")

## 🔑 Step 2: Authenticate with Weights & Biases
Run the cell below and click the link to get your API key from [https://wandb.ai/authorize](https://wandb.ai/authorize).

In [ ]:
# Authenticate your W&B account
wandb.login()

## 📂 Step 3: Load Dataset (`Model_A.csv`)
Upload **`Model_A.csv`** (or `lucas_sentinel2_soil_dataset.csv`) to the Colab Files panel on the left.

In [ ]:
possible_paths = [
    "Model_A.csv",
    "/content/Model_A.csv",
    "lucas_sentinel2_soil_dataset.csv",
    "/content/lucas_sentinel2_soil_dataset.csv"
]

dataset_path = None
for p in possible_paths:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is None:
    raise FileNotFoundError(
        "❌ Dataset file not found! Please upload 'Model_A.csv' or 'lucas_sentinel2_soil_dataset.csv' to Colab Files on the left."
    )

print(f"✅ Loading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path)
print(f"   Total samples: {len(df):,}")
display(df.head(3))

## 🧪 Step 4: Feature Engineering, Target Selection & Spatial Partitioning

In [ ]:
eps = 1e-6
b2 = df["B02"].astype(float)
b3 = df["B03"].astype(float)
b4 = df["B04"].astype(float)
b5 = df["B05"].astype(float)
b6 = df["B06"].astype(float)
b7 = df["B07"].astype(float)
b8 = df["B08"].astype(float)
b8a = df["B8A"].astype(float)
b11 = df["B11"].astype(float)
b12 = df["B12"].astype(float)

# Calculate spectral indices
if "NDVI" not in df.columns:
    df["NDVI"] = (b8 - b4) / (b8 + b4 + eps)
if "BSI" not in df.columns:
    df["BSI"] = ((b11 + b4) - (b8 + b2)) / ((b11 + b4) + (b8 + b2) + eps)
if "Clay_Ratio" not in df.columns:
    df["Clay_Ratio"] = b11 / (b12 + eps)
if "Ferric_Iron" not in df.columns:
    df["Ferric_Iron"] = b4 / (b2 + eps)
if "Carbonate_Index" not in df.columns:
    df["Carbonate_Index"] = b12 / (b11 + eps)
if "NDRE" not in df.columns:
    df["NDRE"] = (b8 - b5) / (b8 + b5 + eps)
if "Soil_Brightness" not in df.columns:
    df["Soil_Brightness"] = np.sqrt((b4**2 + b3**2 + b8**2) / 3.0)

feature_cols = [
    "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12",
    "NDVI", "BSI", "Clay_Ratio", "Ferric_Iron", "Carbonate_Index", "NDRE", "Soil_Brightness"
]
if "Elev" in df.columns:
    feature_cols.append("Elev")

# Target: Soil pH
target_col = "pH_CaCl2" if "pH_CaCl2" in df.columns else "pH"

# Spatial blocks for zero spatial leakage
if "NUTS_2" in df.columns:
    spatial_groups = df["NUTS_2"].astype(str)
    print(f"✅ Spatial blocks (NUTS_2): {spatial_groups.nunique()} unique regional blocks.")
else:
    lat_col = "Latitude" if "Latitude" in df.columns else "TH_LAT"
    lon_col = "Longitude" if "Longitude" in df.columns else "TH_LONG"
    min_lon, min_lat = df[lon_col].min(), df[lat_col].min()
    grid_x = ((df[lon_col] - min_lon) // 2.0).astype(int)
    grid_y = ((df[lat_col] - min_lat) // 2.0).astype(int)
    spatial_groups = grid_x.astype(str) + "_" + grid_y.astype(str)
    print(f"✅ Generated {spatial_groups.nunique()} spatial grid blocks (2° x 2°).")

X_all = df[feature_cols].values
y_all = df[target_col].astype(float).values
print(f"✅ Features: {len(feature_cols)} | Samples: {len(y_all):,} | Target '{target_col}' mean = {y_all.mean():.2f}")

## ⚙️ Step 5: Choose Sweep Configuration (Stage 1 Bayesian or Stage 2 Grid)
You can run either:
- **Option A (Stage 2 - Refinement Grid):** Focused grid search around champion parameters (18 runs).
- **Option B (Stage 1 - Bayesian Optimization):** Wide exploratory search across parameter ranges.

In [ ]:
# Select sweep mode: 'stage2_grid' or 'stage1_bayes'
SWEEP_MODE = 'stage2_grid'  # Set to 'stage1_bayes' for wide search

if SWEEP_MODE == 'stage2_grid':
    # Stage 2: Refinement Grid Search (18 runs = 3 x 3 x 2)
    sweep_config = {
        'name': 'soil-ph-stage2-r2-grid',
        'method': 'grid',
        'metric': {'name': 'val_r2_mean', 'goal': 'maximize'},
        'parameters': {
            'learning_rate': {'values': [0.0225, 0.028, 0.034]},
            'min_child_weight': {'values': [3, 5, 7]},
            'reg_lambda': {'values': [3.2, 4.0]},
            'max_depth': {'value': 7},
            'n_estimators': {'value': 150},
            'reg_alpha': {'value': 0.2},
            'subsample': {'value': 0.8},
            'colsample_bytree': {'value': 0.85}
        }
    }
    DEFAULT_RUN_COUNT = 18
else:
    # Stage 1: Bayesian Exploratory Search
    sweep_config = {
        'name': 'soil-ph-stage1-bayes',
        'method': 'bayes',
        'metric': {'name': 'val_r2_mean', 'goal': 'maximize'},
        'parameters': {
            'max_depth': {'values': [4, 5, 6, 7]},
            'learning_rate': {'distribution': 'uniform', 'min': 0.02, 'max': 0.12},
            'n_estimators': {'values': [150, 250, 350]},
            'subsample': {'distribution': 'uniform', 'min': 0.7, 'max': 0.9},
            'colsample_bytree': {'distribution': 'uniform', 'min': 0.7, 'max': 0.9},
            'reg_alpha': {'distribution': 'uniform', 'min': 0.1, 'max': 1.0},
            'reg_lambda': {'distribution': 'uniform', 'min': 1.0, 'max': 4.5},
            'min_child_weight': {'values': [3, 5, 7]}
        }
    }
    DEFAULT_RUN_COUNT = 20

# Create sweep on W&B
sweep_id = wandb.sweep(sweep_config, project="satellite-soil-ph-tuning")
print(f"🚀 Sweep created in [{SWEEP_MODE}] mode! Sweep ID: {sweep_id}")

## 🔄 Step 6: Define Sweep Training Function with Spatial 5-Fold Cross-Validation

In [ ]:
def train_sweep_iteration():
    with wandb.init() as run:
        cfg = run.config

        gkf = GroupKFold(n_splits=5)
        fold_r2, fold_rmse, fold_mae = [], [], []

        for fold, (train_idx, test_idx) in enumerate(gkf.split(X_all, y_all, groups=spatial_groups)):
            X_tr, y_tr = X_all[train_idx], y_all[train_idx]
            X_te, y_te = X_all[test_idx], y_all[test_idx]

            model = xgb.XGBRegressor(
                max_depth=int(cfg.max_depth),
                learning_rate=float(cfg.learning_rate),
                n_estimators=int(cfg.n_estimators),
                subsample=float(cfg.subsample),
                colsample_bytree=float(cfg.colsample_bytree),
                reg_alpha=float(cfg.reg_alpha),
                reg_lambda=float(cfg.reg_lambda),
                min_child_weight=int(cfg.min_child_weight),
                random_state=42 + fold,
                n_jobs=-1,
                tree_method="hist"
            )
            model.fit(X_tr, y_tr)
            preds = model.predict(X_te)

            fold_r2.append(r2_score(y_te, preds))
            fold_rmse.append(np.sqrt(mean_squared_error(y_te, preds)))
            fold_mae.append(mean_absolute_error(y_te, preds))

        mean_r2 = float(np.mean(fold_r2))
        std_r2 = float(np.std(fold_r2))
        mean_rmse = float(np.mean(fold_rmse))
        std_rmse = float(np.std(fold_rmse))
        mean_mae = float(np.mean(fold_mae))

        # Log metrics to W&B live dashboard
        wandb.log({
            "val_r2_mean": mean_r2,
            "val_r2_std": std_r2,
            "val_rmse_mean": mean_rmse,
            "val_rmse_std": std_rmse,
            "val_mae_mean": mean_mae
        })

print("✅ Spatial cross-validation sweep function ready.")

## 🚀 Step 7: Launch the W&B Sweep Agent
Run the agent below. Click the W&B dashboard link to watch trials log live.

In [ ]:
# Run sweep agent
wandb.agent(sweep_id, function=train_sweep_iteration, count=DEFAULT_RUN_COUNT)

## 🏆 Step 8: Detailed Model Performance, Accuracy Metrics & Optimal Parameters
We query the champion run from W&B, evaluate out-of-fold predictions across all spatial blocks, and display comprehensive performance metrics.

In [ ]:
# 1. Fetch Best Champion Run from W&B
api = wandb.Api()
sweep = api.sweep(f"{wandb.run.entity if wandb.run else api.default_entity}/satellite-soil-ph-tuning/{sweep_id}")
best_run = sweep.best_run()
best_cfg = best_run.config

print("=" * 70)
print(f"🌟 CHAMPION RUN: {best_run.name} (ID: {best_run.id})")
print("=" * 70)

# 2. Re-evaluate Champion Model across 5 Spatial Folds to capture Out-Of-Fold predictions
gkf = GroupKFold(n_splits=5)
oof_predictions = np.zeros(len(y_all))
fold_r2, fold_rmse, fold_mae = [], [], []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_all, y_all, groups=spatial_groups)):
    X_tr, y_tr = X_all[train_idx], y_all[train_idx]
    X_te, y_te = X_all[test_idx], y_all[test_idx]

    m = xgb.XGBRegressor(
        max_depth=int(best_cfg["max_depth"]),
        learning_rate=float(best_cfg["learning_rate"]),
        n_estimators=int(best_cfg["n_estimators"]),
        subsample=float(best_cfg["subsample"]),
        colsample_bytree=float(best_cfg["colsample_bytree"]),
        reg_alpha=float(best_cfg["reg_alpha"]),
        reg_lambda=float(best_cfg["reg_lambda"]),
        min_child_weight=int(best_cfg["min_child_weight"]),
        random_state=42 + fold,
        n_jobs=-1,
        tree_method="hist"
    )
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)
    oof_predictions[test_idx] = p

    fold_r2.append(r2_score(y_te, p))
    fold_rmse.append(np.sqrt(mean_squared_error(y_te, p)))
    fold_mae.append(mean_absolute_error(y_te, p))

overall_r2 = r2_score(y_all, oof_predictions)
overall_rmse = np.sqrt(mean_squared_error(y_all, oof_predictions))
overall_mae = mean_absolute_error(y_all, oof_predictions)
corr, _ = pearsonr(y_all, oof_predictions)

# 3. Display Performance Table
metrics_data = {
    "Metric": [
        "Spatial CV R² Score (Accuracy / Explained Variance)",
        "Spatial CV RMSE (Root Mean Squared Error)",
        "Spatial CV MAE (Mean Absolute Error)",
        "Overall Out-of-Fold R²",
        "Overall Out-of-Fold RMSE",
        "Pearson Correlation (r)"
    ],
    "Value": [
        f"{np.mean(fold_r2):.4f} ± {np.std(fold_r2):.4f}",
        f"{np.mean(fold_rmse):.4f} ± {np.std(fold_rmse):.4f} pH units",
        f"{np.mean(fold_mae):.4f} ± {np.std(fold_mae):.4f} pH units",
        f"{overall_r2:.4f}",
        f"{overall_rmse:.4f} pH units",
        f"{corr:.4f}"
    ]
}
metrics_df = pd.DataFrame(metrics_data)
print("\n📊 ACCURACY & PERFORMANCE METRICS (SPATIAL EVALUATION):")
display(metrics_df)

# 4. Display Optimal Hyperparameters Table
key_params = ["max_depth", "learning_rate", "n_estimators", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda", "min_child_weight"]
params_records = [{"Hyperparameter": p, "Tuned Optimal Value": best_cfg.get(p, "N/A")} for p in key_params]
params_df = pd.DataFrame(params_records)
print("\n⚙️ OPTIMAL TUNED HYPERPARAMETERS:")
display(params_df)

# 5. Retrain Production Model on Full Dataset
final_model = xgb.XGBRegressor(
    max_depth=int(best_cfg["max_depth"]),
    learning_rate=float(best_cfg["learning_rate"]),
    n_estimators=int(best_cfg["n_estimators"]),
    subsample=float(best_cfg["subsample"]),
    colsample_bytree=float(best_cfg["colsample_bytree"]),
    reg_alpha=float(best_cfg["reg_alpha"]),
    reg_lambda=float(best_cfg["reg_lambda"]),
    min_child_weight=int(best_cfg["min_child_weight"]),
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)
final_model.fit(X_all, y_all)

os.makedirs("trained_models", exist_ok=True)
model_save_path = "trained_models/xgboost_pH_finetuned.json"
final_model.save_model(model_save_path)
print(f"\n💾 Successfully saved fine-tuned production model to: {model_save_path}")

## 📈 Step 9: Visual Diagnostics (Actual vs. Predicted Scatter & Feature Importance)
These two publication-quality plots show the accuracy alignment and the most influential satellite bands and physical spectral indices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Actual vs. Predicted Soil pH Scatter Plot
ax1 = axes[0]
sample_idx = np.random.choice(len(y_all), min(1500, len(y_all)), replace=False)
y_sub_true = y_all[sample_idx]
y_sub_pred = oof_predictions[sample_idx]

ax1.scatter(y_sub_true, y_sub_pred, alpha=0.35, color="#2563EB", edgecolors="none", label="Test Predictions")
min_val = min(y_all.min(), oof_predictions.min())
max_val = max(y_all.max(), oof_predictions.max())
ax1.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2, label="1:1 Ideal Reference (y = x)")

# Linear regression trendline
m_fit, b_fit = np.polyfit(y_sub_true, y_sub_pred, 1)
ax1.plot(y_sub_true, m_fit * y_sub_true + b_fit, color="#1E293B", linewidth=1.5, label=f"Fit Line (slope={m_fit:.2f})")

ax1.set_title(f"Soil pH: Actual Lab vs Predicted (Spatial CV R² = {np.mean(fold_r2):.3f})", fontsize=13, fontweight="bold")
ax1.set_xlabel("Measured Ground-Truth Soil pH", fontsize=11)
ax1.set_ylabel("Fine-Tuned XGBoost Predicted pH", fontsize=11)
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.legend(loc="upper left")

# Plot 2: Top 10 Feature Importances
ax2 = axes[1]
importances = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
importances.tail(10).plot(kind="barh", ax=ax2, color="#059669")
ax2.set_title("Top 10 Satellite Features Driving Soil pH Prediction", fontsize=13, fontweight="bold")
ax2.set_xlabel("Feature Importance (Gain)", fontsize=11)
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

## 📥 Step 10: Download the Fine-Tuned Model
Download `xgboost_pH_finetuned.json` directly to your local computer.

In [ ]:
from google.colab import files
files.download("trained_models/xgboost_pH_finetuned.json")
print("✅ Download triggered!")